# Taller 3 — MLflow con el pipeline de scikit-learn

Machine Learning Engineering (CC3105) · UVG · Grupo 6

Este notebook conecta MLflow con el pipeline de Titanic que construimos en las actividades anteriores
(paquete `titanic_pipeline`). No se reescribe el modelo: se reutilizan `extraer_datos`, `limpiar_datos`,
`construir_modelo` y los estimadores registrados como entry points, y MLflow se encarga de registrar
parámetros, métricas, artefactos y el modelo.

Se siguen las dos formas de integración que describe la documentación de MLflow para scikit-learn
([guía de scikit-learn](https://mlflow.org/docs/latest/ml/traditional-ml/sklearn/)):

1. **Autologging** (`mlflow.sklearn.autolog()`): MLflow intercepta `Pipeline.fit()` y registra todos los
   parámetros anidados del pipeline, métricas de entrenamiento, métricas post-entrenamiento y el modelo.
2. **Logging manual** (`mlflow.log_param`, `mlflow.log_metric`, `mlflow.sklearn.log_model`): control total
   sobre qué se registra; útil para comparar los tres modelos del paquete con las mismas métricas.

Después se hace una búsqueda de hiperparámetros con `GridSearchCV` sobre el pipeline completo
(autolog crea un run hijo por combinación), se registra el mejor modelo en el Model Registry y se
carga de vuelta desde MLflow para hacer inferencia.

**Requisitos** (desde la raíz del repositorio):

```bash
pip install -e ".[tracking]"
```

## 1. Configuración de MLflow

In [1]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.models import infer_signature
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split

import titanic_pipeline
from titanic_pipeline.pipeline import (
    FEATURES,
    TARGET,
    construir_modelo,
    extraer_datos,
    limpiar_datos,
    modelos_disponibles,
)

warnings.filterwarnings("ignore")

# Backend local en SQLite: guarda runs, métricas y Model Registry en un solo archivo.
TRACKING_URI = "sqlite:///mlflow.db"
EXPERIMENTO = "taller3-titanic-pipeline"
SEMILLA = 42
# MLflow 3 serializa los modelos de sklearn con skops, que exige declarar los tipos confiables
# (los árboles de RandomForest/GradientBoosting). Los modelos los generamos nosotros, así que es seguro.
TIPOS_CONFIABLES = ["numpy.dtype", "sklearn.tree._tree.Tree"]

mlflow.set_tracking_uri(TRACKING_URI)
experimento = mlflow.set_experiment(EXPERIMENTO)
print("MLflow", mlflow.__version__, "| paquete titanic_pipeline", titanic_pipeline.__version__)
print("Experimento:", experimento.name, "| id:", experimento.experiment_id)
print("Modelos registrados como entry points:", sorted(modelos_disponibles()))

2026/10/13 17:55:02 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/13 17:55:02 INFO mlflow.store.db.utils: Updating database tables


2026/10/13 17:55:03 INFO mlflow.tracking.fluent: Experiment with name 'taller3-titanic-pipeline' does not exist. Creating a new experiment.


MLflow 3.17.1 | paquete titanic_pipeline 1.4.0
Experimento: taller3-titanic-pipeline | id: 1
Modelos registrados como entry points: ['gradient_boosting', 'logistic_regression', 'random_forest']


## 2. Datos

Mismas etapas y misma partición que `entrenar_modelo` del paquete (80/20, estratificada, `random_state=42`),
para que las métricas sean comparables con las de la Actividad 9.

In [2]:
datos = limpiar_datos(extraer_datos())
X = datos[FEATURES]
y = datos[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEMILLA, stratify=y
)
print(f"Train: {X_train.shape} | Test: {X_test.shape} | Tasa de supervivencia: {y.mean():.3f}")
X_train.head()

Train: (712, 5) | Test: (179, 5) | Tasa de supervivencia: 0.384


,Pclass,Sex,Age,Fare,Embarked
692,3,male,NaN,56.4958,s
481,2,male,NaN,0.0000,s
527,1,male,NaN,221.7792,s
855,3,female,18.0,9.3500,s
801,2,female,31.0,26.2500,s


## 3. Run 1 — Autologging del pipeline

Con `autolog()` basta llamar `fit()`. MLflow registra:

- Todos los parámetros del `Pipeline`, incluidos los anidados (`preprocesamiento__num__imputer__strategy`,
  `modelo__n_estimators`, etc.), porque internamente usa `get_params(deep=True)`.
- Métricas de entrenamiento (`training_accuracy_score`, `training_f1_score`, ...).
- Métricas post-entrenamiento: llamadas a `model.score(...)` después del `fit` (p. ej. `Pipeline_score_X_test`).
  La documentación indica que también se capturan llamadas a `sklearn.metrics`, pero en nuestra prueba con
  scikit-learn 1.9 solo se registró `.score()`; por eso las métricas de test también se registran explícitamente.
- El modelo serializado, con firma (*signature*) y ejemplo de entrada.

In [3]:
mlflow.sklearn.autolog(log_input_examples=True, log_model_signatures=True, silent=True)

with mlflow.start_run(run_name="autolog_random_forest") as run_autolog:
    mlflow.set_tags({"tipo": "autolog", "grupo": "Grupo 6", "curso": "CC3105"})
    pipeline_rf = construir_modelo("random_forest")
    pipeline_rf.fit(X_train, y_train)

    pipeline_rf.score(X_test, y_test)  # autolog la registra como Pipeline_score_X_test
    pred = pipeline_rf.predict(X_test)
    acc, f1 = accuracy_score(y_test, pred), f1_score(y_test, pred)
    mlflow.log_metrics({"test_accuracy": acc, "test_f1": f1})

print(f"Accuracy test: {acc:.4f} | F1 test: {f1:.4f}")
run = mlflow.get_run(run_autolog.info.run_id)
print(f"Parámetros registrados automáticamente: {len(run.data.params)}")
# Algunos de los parámetros anidados que autolog capturó del pipeline:
pd.Series(run.data.params).filter(
    regex=r"(imputer__strategy|encoder__handle_unknown|modelo__(n_estimators|max_depth|min_samples_leaf|random_state))$"
)

Accuracy test: 0.8212 | F1 test: 0.7576
Parámetros registrados automáticamente: 69


preprocesamiento__num__imputer__strategy                 median
preprocesamiento__cat__imputer__strategy          most_frequent
preprocesamiento__cat__encoder__handle_unknown           ignore
modelo__max_depth                                          None
modelo__min_samples_leaf                                      1
modelo__n_estimators                                        200
modelo__random_state                                         42
dtype: object

In [4]:
pd.Series(run.data.metrics).sort_index()

Pipeline_score_X_test       0.821229
test_accuracy               0.821229
test_f1                     0.757576
training_accuracy_score     0.981742
training_f1_score           0.981681
training_log_loss           0.117664
training_precision_score    0.981956
training_recall_score       0.981742
training_roc_auc            0.997447
training_score              0.981742
dtype: float64

## 4. Run 2 — Comparación de los modelos del paquete (logging manual)

Se desactiva el autolog para tener control explícito. Se crea un run padre y un run hijo (*nested run*)
por cada estimador descubierto vía entry points. En cada hijo se registran:

- Parámetros: nombre del modelo e hiperparámetros del estimador.
- Métricas en test (accuracy, precision, recall, F1, ROC AUC) y accuracy con validación cruzada de 5 folds en train.
- Artefactos: matriz de confusión (figura) y `classification_report` (JSON).
- El dataset de entrenamiento como *input* del run y el pipeline completo con firma y ejemplo de entrada.

In [5]:
mlflow.sklearn.autolog(disable=True)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
dataset_train = mlflow.data.from_pandas(
    pd.concat([X_train, y_train], axis=1), name="titanic_train", targets=TARGET
)


def evaluar_y_registrar(nombre_modelo: str) -> dict:
    pipeline = construir_modelo(nombre_modelo)
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="accuracy")
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    proba = pipeline.predict_proba(X_test)[:, 1]

    metricas = {
        "test_accuracy": accuracy_score(y_test, pred),
        "test_precision": precision_score(y_test, pred),
        "test_recall": recall_score(y_test, pred),
        "test_f1": f1_score(y_test, pred),
        "test_roc_auc": roc_auc_score(y_test, proba),
        "cv_accuracy_mean": cv_scores.mean(),
        "cv_accuracy_std": cv_scores.std(),
    }

    mlflow.log_input(dataset_train, context="training")
    mlflow.log_param("modelo", nombre_modelo)
    mlflow.log_params({f"modelo__{k}": v for k, v in pipeline.named_steps["modelo"].get_params().items()})
    mlflow.log_metrics(metricas)

    fig, ax = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, colorbar=False)
    ax.set_title(nombre_modelo)
    mlflow.log_figure(fig, "matriz_confusion.png")
    plt.close(fig)
    mlflow.log_dict(classification_report(y_test, pred, output_dict=True), "classification_report.json")

    model_info = mlflow.sklearn.log_model(
        sk_model=pipeline,
        name="modelo",
        signature=infer_signature(X_test, pred),
        input_example=X_test.head(5),
        skops_trusted_types=TIPOS_CONFIABLES,
    )
    return {"modelo": nombre_modelo, **metricas, "model_uri": model_info.model_uri}


resultados = []
with mlflow.start_run(run_name="comparacion_modelos") as padre:
    mlflow.set_tags({"tipo": "comparacion", "grupo": "Grupo 6", "curso": "CC3105"})
    for nombre in sorted(modelos_disponibles()):
        with mlflow.start_run(run_name=nombre, nested=True):
            resultados.append(evaluar_y_registrar(nombre))

comparacion = pd.DataFrame(resultados).set_index("modelo").sort_values("test_f1", ascending=False)
comparacion.drop(columns="model_uri").round(4)

,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,cv_accuracy_mean,cv_accuracy_std
modelo,,,,,,,
random_forest,0.8212,0.7937,0.7246,0.7576,0.8402,0.8118,0.0214
gradient_boosting,0.8156,0.8214,0.6667,0.7360,0.8212,0.8188,0.0302
logistic_regression,0.7709,0.7188,0.6667,0.6917,0.8341,0.7936,0.0165


## 5. Run 3 — Búsqueda de hiperparámetros con `GridSearchCV` sobre el pipeline

La ventaja de tener todo dentro de un `Pipeline` es que la búsqueda puede incluir tanto el preprocesamiento
como el modelo (sintaxis `paso__subpaso__parametro`), y el preprocesamiento se ajusta dentro de cada fold
(sin *data leakage*). Con autolog activo, MLflow crea un run padre con el mejor modelo y un run hijo por
cada combinación evaluada.

In [6]:
mlflow.sklearn.autolog(log_input_examples=True, max_tuning_runs=None, silent=True)

param_grid = {
    "preprocesamiento__num__imputer__strategy": ["median", "mean"],
    "modelo__n_estimators": [100, 300],
    "modelo__max_depth": [None, 5, 8],
    "modelo__min_samples_leaf": [1, 3],
}
busqueda = GridSearchCV(
    construir_modelo("random_forest"), param_grid, cv=cv, scoring="f1", n_jobs=-1, refit=True
)

with mlflow.start_run(run_name="gridsearch_random_forest") as run_grid:
    mlflow.set_tags({"tipo": "gridsearch", "grupo": "Grupo 6", "curso": "CC3105"})
    busqueda.fit(X_train, y_train)
    pred_grid = busqueda.predict(X_test)
    f1_grid, acc_grid = f1_score(y_test, pred_grid), accuracy_score(y_test, pred_grid)
    mlflow.log_metrics({"test_accuracy": acc_grid, "test_f1": f1_grid})

mlflow.sklearn.autolog(disable=True)
print("Mejores parámetros:", busqueda.best_params_)
print(f"F1 CV: {busqueda.best_score_:.4f} | F1 test: {f1_grid:.4f} | Accuracy test: {acc_grid:.4f}")

Mejores parámetros: {'modelo__max_depth': None, 'modelo__min_samples_leaf': 1, 'modelo__n_estimators': 300, 'preprocesamiento__num__imputer__strategy': 'median'}
F1 CV: 0.7515 | F1 test: 0.7481 | Accuracy test: 0.8156


**Lectura.** En nuestra ejecución, la combinación ganadora del grid coincide casi con la configuración por defecto
(300 árboles en lugar de 200, sin límite de profundidad) y su F1 en test (≈0.748) no supera al Random Forest base
(≈0.758): la diferencia está dentro del ruido de un test de 179 filas. El autolog del run 1 también muestra
accuracy de ≈0.98 en entrenamiento contra ≈0.82 en test, es decir, el bosque memoriza el train. Tener las
24 combinaciones registradas en MLflow permite ver esto en la vista *Compare* en lugar de quedarse solo con
`best_params_`.

## 6. Consultar los runs desde código

`mlflow.search_runs` devuelve un DataFrame con todos los runs del experimento; es lo mismo que se ve en
la tabla de la UI y permite elegir el mejor modelo de forma programática.

In [7]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENTO],
    filter_string="metrics.test_f1 > 0",
    order_by=["metrics.test_f1 DESC"],
)
runs[["tags.mlflow.runName", "metrics.test_accuracy", "metrics.test_f1",
      "metrics.test_roc_auc", "metrics.cv_accuracy_mean"]].round(4)

,tags.mlflow.runName,metrics.test_accuracy,metrics.test_f1,metrics.test_roc_auc,metrics.cv_accuracy_mean
0,random_forest,0.8212,0.7576,0.8402,0.8118
1,autolog_random_forest,0.8212,0.7576,NaN,NaN
2,gridsearch_random_forest,0.8156,0.7481,NaN,NaN
3,gradient_boosting,0.8156,0.7360,0.8212,0.8188
4,logistic_regression,0.7709,0.6917,0.8341,0.7936


## 7. Registrar el modelo final en el Model Registry

Se toma el mejor pipeline del `GridSearchCV`, se evalúa con las mismas métricas de la sección 4 y se registra
como versión del modelo `titanic-pipeline`. Si el F1 en test no supera al mejor modelo de la comparación,
se registra ese en su lugar. En ambos casos el run `modelo_final` queda con sus métricas y la versión
registrada apunta a él.

In [8]:
mejor_base = comparacion.iloc[0]
usar_grid = f1_grid >= mejor_base["test_f1"]

with mlflow.start_run(run_name="modelo_final"):
    mlflow.set_tags({"tipo": "modelo_final", "grupo": "Grupo 6", "curso": "CC3105"})
    if usar_grid:
        final = busqueda.best_estimator_
        mlflow.log_params({"origen": "gridsearch_random_forest", **busqueda.best_params_})
    else:
        final = construir_modelo(mejor_base.name).fit(X_train, y_train)
        mlflow.log_param("origen", f"comparacion_modelos/{mejor_base.name}")

    pred = final.predict(X_test)
    proba = final.predict_proba(X_test)[:, 1]
    mlflow.log_metrics({
        "test_accuracy": accuracy_score(y_test, pred),
        "test_precision": precision_score(y_test, pred),
        "test_recall": recall_score(y_test, pred),
        "test_f1": f1_score(y_test, pred),
        "test_roc_auc": roc_auc_score(y_test, proba),
    })
    info = mlflow.sklearn.log_model(
        sk_model=final,
        name="modelo",
        signature=infer_signature(X_test, pred),
        input_example=X_test.head(5),
        registered_model_name="titanic-pipeline",
        skops_trusted_types=TIPOS_CONFIABLES,
    )

version = info.registered_model_version
print(f"Origen: {'GridSearchCV' if usar_grid else mejor_base.name} | titanic-pipeline v{version}")

Origen: random_forest | titanic-pipeline v1


Successfully registered model 'titanic-pipeline'.
Created version '1' of model 'titanic-pipeline'.


## 8. Cargar el modelo desde MLflow e inferir

El modelo se carga desde el registry con la URI `models:/<nombre>/<versión>`, sin depender de rutas locales
ni de `joblib`. Se usa la interfaz genérica `pyfunc`, que es la misma que usaría `mlflow models serve`.
Los pasajeros nuevos pasan por la misma limpieza del paquete (como hace `titanic-predict`).

In [9]:
modelo_cargado = mlflow.pyfunc.load_model(f"models:/titanic-pipeline/{version}")

nuevos = pd.DataFrame({
    "Pclass": [1, 3, 2],
    "Sex": ["Female", "male ", "male"],
    "Age": [29.0, 22.0, None],
    "Fare": [211.34, 7.25, 13.0],
    "Embarked": ["S", "Q", None],
})
entrada = limpiar_datos(nuevos.assign(Survived=0))[FEATURES]
nuevos.assign(Survived_Pred=modelo_cargado.predict(entrada))

,Pclass,Sex,Age,Fare,Embarked,Survived_Pred
0,1,Female,29.0,211.34,S,1
1,3,male,22.0,7.25,Q,0
2,2,male,NaN,13.00,None,0


## 9. Ver los resultados en la UI de MLflow

Desde la carpeta `notebooks/` (donde quedó `mlflow.db`):

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

y abrir <http://127.0.0.1:5000>. Capturas sugeridas para el entregable:

1. Tabla del experimento `taller3-titanic-pipeline` con los runs y columnas de métricas.
2. Detalle del run `autolog_random_forest`: parámetros anidados del pipeline y métricas registradas solas.
3. Run `comparacion_modelos` expandido con sus tres runs hijos, y la vista *Compare* (gráfica de `test_f1` por modelo).
4. Artefactos de un run hijo: `matriz_confusion.png`, `classification_report.json` y el modelo con su firma.
5. Runs hijos del `gridsearch_random_forest`.
6. *Models* → `titanic-pipeline` con la versión registrada.